# Import Statements

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bert-weights/bert-1/config.json
/kaggle/input/bert-weights/bert-1/tokenizer.json
/kaggle/input/bert-weights/bert-1/tokenizer_config.json
/kaggle/input/bert-weights/bert-1/model.safetensors
/kaggle/input/bert-weights/bert-1/special_tokens_map.json
/kaggle/input/bert-weights/bert-1/vocab.txt
/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv
/kaggle/input/2025-sep-dl-gen-ai-project/train.csv
/kaggle/input/2025-sep-dl-gen-ai-project/test.csv
/kaggle/input/bi-lstm-attention/bilstm_att_dls_model.pth
/kaggle/input/bi-lstm-attention/Bi_LSTM_Model_dls_vocab.json
/kaggle/input/distilbert-weights/distilbert-1/config.json
/kaggle/input/distilbert-weights/distilbert-1/tokenizer.json
/kaggle/input/distilbert-weights/distilbert-1/tokenizer_config.json
/kaggle/input/distilbert-weights/distilbert-1/model.safetensors
/kaggle/input/distilbert-weights/distilbert-1/special_tokens_map.json
/kaggle/input/distilbert-weights/distilbert-1/vocab.txt
/kaggle/input/roberta-weights/robert

In [2]:
# !pip install contractions

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from tqdm.auto import tqdm
import json

# Test Dataset

In [28]:
df_test = pd.read_csv('/kaggle/input/processed/test_lemmastop.csv')
df_test['processed_text'] = df_test['processed_text'].fillna('')
df_test.head()

,id,text,clean_text,processed_text
0,0,she wanted to fight over every single little t...,she wanted to fight over every single little t...,wanted fight every single little thing
1,1,"anyway, back to tuesday.",anyway back to tuesday,anyway back tuesday
2,2,she shrieked at the dog to go back.,she shrieked at the dog to go back,shrieked dog go back
3,3,yelling for everyone to get back or get inside...,yelling for everyone to get back or get inside...,yelling everyone get back get inside draw knif...
4,4,still kind of freaky.,still kind of freaky,still kind freaky


In [29]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']

# Set Device

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Tokenization

In [31]:
def text_to_sequence(text, vocab):
    """Converts a text string to a sequence of integers using the vocab."""
    tokens = word_tokenize(text)
    return [vocab.get(word, vocab.get('<UNK>', 1)) for word in tokens]

In [32]:
def pad_sequence(seq, max_len):
    """Pads a sequence to max_len. Truncates if longer."""
    if len(seq) > max_len:
        return seq[:max_len]  # Truncate
    else:
        return seq + [vocab.get('<PAD>', 0)] * (max_len - len(seq))

# Bi-LSTM

In [40]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers, 
            bidirectional=True, 
            dropout=dropout,
            batch_first=True
        )
        
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text shape: [batch_size, seq_len]
        embedded = self.embedding(text)
        # embedded shape: [batch_size, seq_len, embedding_dim]
        
        # outputs shape: [batch_size, seq_len, hidden_dim * 2]
        # hidden shape: [n_layers * 2, batch_size, hidden_dim]
        outputs, (hidden, cell) = self.lstm(embedded)
        
        # Concatenate the final forward and backward hidden states
        # hidden[-2,:,:] is the final forward hidden state
        # hidden[-1,:,:] is the final backward hidden state
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        # hidden shape: [batch_size, hidden_dim * 2]

        hidden = self.dropout(hidden)
        prediction = self.fc(hidden)
        # prediction shape: [batch_size, output_dim]
        
        return prediction

# Attention-based Bi-LSTM

In [41]:
class LSTMAttentionModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers, 
            bidirectional=True, 
            dropout=dropout, 
            batch_first=True
        )
        
        # The Attention Layer
        self.attention = nn.Linear(hidden_dim * 2, 1)
        
        # The Final Classification Layer
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text: [batch_size, seq_len]
        embedded = self.embedding(text)
        
        # output: [batch_size, seq_len, hidden_dim * 2]
        output, (hidden, cell) = self.lstm(embedded)
        
        # ATTENTION MECHANISM
        # energy: [batch_size, seq_len, 1]
        energy = torch.tanh(self.attention(output))
        
        # weights: [batch_size, seq_len, 1]
        weights = F.softmax(energy, dim=1)
        
        # weighted: [batch_size, seq_len, hidden_dim * 2]
        weighted = output * weights
        
        # context_vector: [batch_size, hidden_dim * 2]
        context_vector = torch.sum(weighted, dim=1)
        
        # Pass the Context Vector to the classifier
        final_input = self.dropout(context_vector)
        prediction = self.fc(final_input)
        
        return prediction

In [42]:
# os.environ["WANDB_DISABLED"] = "true"

# Parameters

In [43]:
MAX_LEN = 18
BATCH_SIZE = 32
EMBEDDING_DIM = 100 
HIDDEN_DIM = 128     
OUTPUT_DIM = 5      
N_LAYERS = 2         
DROPOUT = 0.4        
LEARNING_RATE = 1e-3
N_EPOCHS = 50

# Load Vocab and Model

In [44]:
base_path = "/kaggle/input/bi-lstm-attention" 

with open(f'{base_path}/Bi_LSTM_Model_dls_vocab.json', 'r') as f:
    vocab = json.load(f)

model = LSTMAttentionModel(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT)
model.load_state_dict(torch.load(f'{base_path}/bilstm_att_dls_model.pth', map_location=device))
model = model.to(device)
model.eval()

LSTMAttentionModel(
  (embedding): Embedding(4873, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (attention): Linear(in_features=256, out_features=1, bias=True)
  (fc): Linear(in_features=256, out_features=5, bias=True)
  (dropout): Dropout(p=0.4, inplace=False)
)

In [26]:
sequences = [text_to_sequence(text, vocab) for text in df_test['processed_text']]
test_sequences = [pad_sequence(s, MAX_LEN) for s in sequences]
X_test = torch.tensor(test_sequences, dtype=torch.long)

test_data = torch.utils.data.TensorDataset(X_test)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

# Make Predictions

In [18]:
all_preds = []

print("Starting predictions.")
with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)
        
        logits = model(inputs)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()
        
        all_preds.extend(preds.cpu().numpy())

Starting predictions.


# Submission

In [19]:
df_submission = pd.DataFrame(all_preds, columns=emotion_cols)
df_submission.insert(0, 'id', df_test['id'])
df_submission.head()

,id,anger,fear,joy,sadness,surprise
0,0,1,0,1,0,0
1,1,0,0,0,0,0
2,2,1,1,0,0,0
3,3,0,1,0,0,0
4,4,0,1,0,0,1


In [20]:
df_submission.to_csv('submission.csv', index=False)